# 🚴‍♂️ Strava Activities Collector

This notebook reads the access tokens of each authorized athlete and collects their latest activities from Strava using the API.

The activities are combined into a single CSV file for further analysis.

---

### ✅ What this notebook does

1. Loads the list of authorized athletes from `tokens_atletas.csv`
2. Uses their access tokens to collect recent activities (max 100)
3. Combines all data into a single DataFrame
4. Saves everything in a CSV file: `atividades_todos.csv`

> ⚠️ Only visible activities (type = "Ride") are considered for now.


### 1. 📦 Load Required Libraries

In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

### 2. ⚙️ Load Environment Variables

In [2]:
# Load environment variables
load_dotenv()

TOKENS_PATH = TOKENS_PATH = r"C:\Users\dsgal\Documents\Activities\data\tokens_athletes.csv"
ACTIVITIES_PATH = os.getenv("ACTIVITIES_PATH")

### 3. 🧾 Load tokens CSV

In [3]:
# Load athlete tokens
df_tokens = pd.read_csv(TOKENS_PATH)
activities_all = []

### 4. 🔁Fetch activities from Strava

In [4]:
for _, row in df_tokens.iterrows():
    name = row["nome"]
    token = row["access_token"]

    print(f"🔍 Collecting activities for {name}...")

    headers = {"Authorization": f"Bearer {token}"}

    page = 1
    per_page = 200
    all_activities = []

    while True:
        url = "https://www.strava.com/api/v3/athlete/activities"
        params = {"per_page": per_page, "page": page}
        resp = requests.get(url, headers=headers, params=params)

        if resp.status_code != 200:
            print(f"⚠️ Failed to collect activities for {name}: {resp.status_code} {resp.text}")
            break

        batch = resp.json()
        if not batch:  # página vazia => acabou
            break

        all_activities.extend(batch)
        page += 1

    if not all_activities:
        print(f"⚠️ No activities found for {name}")
        continue

    df_activities = pd.json_normalize(all_activities)
    df_activities["nome"] = name
    activities_all.append(df_activities)

    print(f"✅ Done collecting activities for {name}: {len(all_activities)} activities")

🔍 Collecting activities for Diego Galdino...
✅ Done collecting activities for Diego Galdino: 980 activities


In [ ]:
# for index, row in df_tokens.iterrows():
#     name = row['nome']
#     token = row['access_token']

#     print(f"🔍 Collecting activities for {name}...")

#     headers = {'Authorization': f'Bearer {token}'}
#     response = requests.get('https://www.strava.com/api/v3/athlete/activities?per_page=100', headers=headers)

#     if response.status_code != 200:
#         print(f"⚠️ Failed to collect activities for {name}: {response.status_code}")
#         continue

#     activities = response.json()

#     if not activities:
#         print(f"⚠️ No activities found for {name}")
#         continue

#     df_activities = pd.json_normalize(activities)
#     df_activities['nome'] = name
#     activities_all.append(df_activities)
#     print(f"✅ Done collecting activities for {name}")


🔍 Collecting activities for Diego Galdino...
✅ Done collecting activities for Diego Galdino


### 5. 📥Combine and Save Data

In [5]:
if activities_all:
    df_all = pd.concat(activities_all, ignore_index=True)
    df_all.to_csv("data/activities_all.csv", index=False)
    print(f"✅ All activities saved to: ")
else:
    print("❌ No activities collected.")

✅ All activities saved to: 
